# 81 字实验 · 会话恢复与分段续训

本文件根据你从 Colab 导出的当前状态整理，更新于 **2026-10-13**。
日常只需下方 **3 个代码单元**；完整代码已嵌入，无需上传 `session_cells` 目录。

| 已知实验状态 | 当前处理 |
|---|---|
| smoke、小集拟合、40 轮基线 | 已完成，不在初始化时重跑 |
| 第 1 项：学习过程复盘 | 已有结果；第③格读取并复核日志 |
| 第 2 项：分类间隔 | 已有结果；第③格读取保存的报告 |
| 第 3 项：学习曲线 | 已确认四档准备成功；训练进度以 Drive 实际文件为准 |
| 第 6 项：训练效率与资源分析 | 第③格复用已有三轮计时完成分析；只有实际校验成功才标记完成 |

**这三个单元的编号与上述分析项目编号是两套编号。**

原来的 29 个单元、图片、输出及历史修复代码，逐字节保存在同目录的
`hanzi81_baseline_new.before_session_refactor.ipynb`。该副本供追溯，不应“全部运行”。
当前文件不携带旧输出，以免把历史结果误认成本次执行结果。
原基线 `train.py`、学习曲线脚本、数据和 ZIP 均保持原版本。

**仅查看结果：** 使用 CPU 运行时，直接运行第③格；它会自行挂载 Drive。

**完成第 6 项：** 同样直接运行第③格。它会校验已有计时报告，计算训练/验证吞吐、
阶段耗时占比、显存容量占比、检查点大小和训练时长估算，并保存 `efficiency_resources.json`
与图表。无需重新进行三轮 GPU 计时；缺失或损坏的报告会保留未完成状态。

**继续学习曲线：** 连接与原实验匹配的 GPU → 运行第①格 → 在第②格显式开启训练，
选择一个档位和本次块数 → 运行第③格检查保存。
默认训练关闭；默认每次只给一个档位追加 3 块。分段不改变每档 10 块 / 2430 批次的总预算。

Drive 项目仍为 `/content/drive/MyDrive/hanzi81`。如确需换目录，要同步修改相关单元顶部的
`DRIVE_ROOT`。本文件面向已建立的实验；不会自动重建缺失的实验协议或放宽环境检查。


## ① 初始化与恢复

合并 Drive 挂载、路径、固定 ZIP 恢复、文件校验、学习曲线脚本恢复和 GPU 环境检查。
重复运行时复用已校验的本地文件；不用重新上传辅助脚本，也不升级 torch/torchvision。
代码与数据位于 `/content`，检查点继续保存在 Drive。

原环境：Python 3.13.15、torch 2.11.0+cu130、torchvision 0.26.0+cu130、NumPy 2.1.3、Tesla T4。
版本或 GPU 改变时严格续训会停止并显示差异；不要删除检查来强行续训。
`PREPARE_TRAINING=False` 时只挂载 Drive；看报告也可以直接跳到第③格。


In [ ]:
#@title ① 初始化与恢复（不会启动训练）
from pathlib import Path
import hashlib
import json
import shutil
import stat
import subprocess
import sys
import zipfile

PREPARE_TRAINING = True #@param {type:"boolean"}
DRIVE_ROOT = Path("/content/drive/MyDrive/hanzi81")
LOCAL_ROOT = Path("/content")

BUNDLE_SHA256 = "e477bdd4894e6b585a2149d6f22db3948e5d6e20075d532fded5171d1b57dd35"
BUNDLE_MANIFEST_SHA256 = "42004e354a1e70a7325f82b7c5752842698caf6476543eaa149f494db6113ec5"
CURVE_SHA256 = "2b098c465e94fd1909c159c1a64b59745e914c13023e1a23d8a3445e2231272d"
SUITE_RELATIVE = "analyses/baseline81_s42_v1/03_learning_curve/steps2430_s42_v1"


def setup_digest(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def setup_mount(root):
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive
        drive.mount("/content/drive")
    if not root.is_dir():
        raise FileNotFoundError(f"项目目录不存在，请检查 Drive 挂载和 DRIVE_ROOT：{root}")


def setup_verify_work(work):
    manifest_path = work / "bundle_manifest.json"
    if not manifest_path.is_file():
        return False
    if setup_digest(manifest_path) != BUNDLE_MANIFEST_SHA256:
        raise ValueError("本地包清单与原实验不同；保留现场，恢复原始包，不绕过校验。")
    payload = json.loads(manifest_path.read_text(encoding="utf-8"))
    missing = False
    for name, expected in payload["files"].items():
        path = (work / name).resolve()
        if not path.is_relative_to(work.resolve()):
            raise ValueError("包清单包含非法路径")
        if not path.is_file():
            missing = True
        elif setup_digest(path) != expected:
            raise ValueError(f"本地文件校验失败：{name}；未覆盖此文件，请先检查原因。")
    return not missing


def setup_verified_copy(source, target, expected):
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_name(target.name + ".copying")
    shutil.copyfile(source, temporary)
    if setup_digest(temporary) != expected:
        raise ValueError(f"复制内容校验失败，未替换目标：{target}")
    temporary.replace(target)


def setup_restore_bundle(root, local_root):
    work = local_root / ("hanzi81-" + BUNDLE_SHA256[:12])
    if setup_verify_work(work):
        print("复用已校验的本地代码与数据：", work)
        return work
    local_zip = local_root / "hanzi81_colab_bundle.zip"
    archive_snapshot = root / "packages" / (BUNDLE_SHA256 + ".zip")
    if not local_zip.is_file() or setup_digest(local_zip) != BUNDLE_SHA256:
        candidates = [root / "hanzi81_colab_bundle.zip", archive_snapshot]
        source = next((p for p in candidates if p.is_file()), None)
        if source is None:
            raise FileNotFoundError(f"请恢复原始上传包：{candidates[0]}")
        setup_verified_copy(source, local_zip, BUNDLE_SHA256)
    with zipfile.ZipFile(local_zip) as archive:
        for entry in archive.infolist():
            destination = (work / entry.filename).resolve()
            if not destination.is_relative_to(work.resolve()) or stat.S_ISLNK(entry.external_attr >> 16):
                raise ValueError("ZIP 包含非法路径或符号链接")
        # ZIP 哈希与固定原包一致，缺失的运行时文件可以从它恢复。
        work.mkdir(parents=True, exist_ok=True)
        archive.extractall(work)
    if not setup_verify_work(work):
        raise ValueError("解包后仍有文件缺失")
    if not archive_snapshot.exists():
        setup_verified_copy(local_zip, archive_snapshot, BUNDLE_SHA256)
    print("已从固定原包恢复：", work)
    return work


def setup_restore_curve(root, local_root):
    suite = root / SUITE_RELATIVE
    plan_path, source = suite / "plan.json", suite / "learning_curve.py"
    if not plan_path.is_file() or not source.is_file():
        raise FileNotFoundError(
            f"找不到已准备的学习曲线协议/脚本：{suite}。请检查 Drive 路径；本单元不会另建实验。")
    plan = json.loads(plan_path.read_text(encoding="utf-8"))
    if plan["curve_script_sha256"] != CURVE_SHA256 or setup_digest(source) != CURVE_SHA256:
        raise ValueError("Drive 中的学习曲线脚本与冻结版本不一致")
    if (plan["sizes"], plan["blocks"], plan["block_draws"], plan["batch_size"], plan["seed"]) != (
            [24, 48, 96, 192], 10, 15552, 64, 42):
        raise ValueError("学习曲线预算或种子与当前实验不同")
    target = local_root / "hanzi81_learning_curve.py"
    if not target.is_file() or setup_digest(target) != CURVE_SHA256:
        setup_verified_copy(source, target, CURVE_SHA256)
    print("学习曲线脚本已恢复；每档总预算仍为 10 块 / 2430 批次。")
    return target


def setup_probe(work, reference):
    # 在子进程中检查 GPU；不向 Notebook 主进程长期保留 CUDA 上下文或模型。
    code = """
import json
from pathlib import Path
import torch
from training import train as T
cfg = json.loads(Path(__import__('sys').argv[1]).read_text(encoding='utf-8'))
if not torch.cuda.is_available():
    raise RuntimeError('未分配 GPU；只读报告可直接运行第③格，无需 GPU。')
actual = T.environment(torch.device('cuda'))
print(json.dumps(actual, ensure_ascii=False, indent=2), flush=True)
if actual != cfg['environment']:
    difference = {k: {'saved': cfg['environment'].get(k), 'current': actual.get(k)}
                  for k in set(actual) | set(cfg['environment'])
                  if actual.get(k) != cfg['environment'].get(k)}
    raise RuntimeError('环境与原实验不同，禁止直接续训：' + json.dumps(difference, ensure_ascii=False))
with torch.inference_mode():
    probe = torch.nn.Conv2d(1, 8, 3).cuda()
    print('GPU 前向检查:', tuple(probe(torch.zeros(2, 1, 64, 64, device='cuda')).shape))
"""
    subprocess.run([sys.executable, "-u", "-c", code, str(reference)], cwd=work, check=True)


def initialize_session(root, local_root, prepare_training):
    setup_mount(root)
    if not prepare_training:
        print("Drive 已连接；可直接运行第③格。未导入 torch、复制数据或执行 GPU 检查。")
        return
    work = setup_restore_bundle(root, local_root)
    data = work / "training/data/baseline81-v2"
    manifest = json.loads((data / "manifest.json").read_text(encoding="utf-8"))
    counts = {k: v["images"] for k, v in manifest["splits"].items()}
    if counts != {"train": 15552, "val_select": 7776, "val_graph": 7776, "test": 15552}:
        raise ValueError("数据规模与原实验不同")
    print("四份文件数量校验：", counts, "（未执行测试集推理）")
    setup_restore_curve(root, local_root)
    reference = root / "runs/baseline81_s42_v1/config.json"
    if not reference.is_file():
        raise FileNotFoundError(f"缺少原基线配置：{reference}")
    cfg = json.loads(reference.read_text(encoding="utf-8"))
    for key, path in [("trainer_sha256", work / "training/train.py"),
                      ("io_sha256", work / "training/io_utils.py"),
                      ("data_sha256", data / "manifest.json")]:
        if cfg[key] != setup_digest(path):
            raise ValueError(f"原基线配置与源文件不同：{key}")
    setup_probe(work, reference)
    print("初始化完成。第②格默认关闭；需续训时显式开启。")
    print("第 6 项可在第③格复用已有报告完成分析；本次初始化不会重跑计时。")


if __name__ == "__main__":
    initialize_session(DRIVE_ROOT, LOCAL_ROOT, PREPARE_TRAINING)


## ② 开始或继续一个学习曲线档位

只有 `RUN_TRAINING=True` 才会启动训练。每字图数 `PER_CLASS` 可选 192、96、48、24；
建议按这个顺序完成，但各档互相独立。`BLOCKS_THIS_SESSION` 是**本次追加块数**，不是总预算。
例如已保存 3 块、再运行 3 块，就续至第 6 块；已完成 10 块的档位由原脚本校验后跳过。

每块 243 批次，保存模型、优化器、学习率计划、AMP scaler 和随机状态。
中断后恢复最近完整块；未保存的半块需要重跑。一次只运行一个写入该档位的会话。
输出实时显示，并额外保存在实验目录的 `session_logs` 中。

本格拥有自己的导入和路径，不再依赖 `run_learning_curve` 等上一格定义的函数。
缺少本地数据时会明确提示回第①格恢复。四档尚未全部完成时，正常分段结束不会报错。
基线旧的 40 轮结果不充当学习曲线 192 图档位；原实验参数和总训练量均保持不变。


In [ ]:
#@title ② 分段续训学习曲线（默认关闭，一次一个档位）
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import subprocess
import sys
import uuid

RUN_TRAINING = False #@param {type:"boolean"}
PER_CLASS = 192 #@param [192, 96, 48, 24] {type:"raw"}
BLOCKS_THIS_SESSION = 3 #@param {type:"integer"}
DRIVE_ROOT = Path("/content/drive/MyDrive/hanzi81")
LOCAL_ROOT = Path("/content")

TRAIN_CURVE_SHA256 = "2b098c465e94fd1909c159c1a64b59745e914c13023e1a23d8a3445e2231272d"
TRAIN_SOURCE_SHA256 = "f31aece79c2322772139f2f299aa88dc272ec7d02ef3308bcaeeeb48e8cd4197"
TRAIN_IO_SHA256 = "6e0effced2c2a35bd474fba9925404bf84182d8c39348e365052bc7d0d9cb183"
TRAIN_DATA_SHA256 = "6a2d237ae03ed219f76c2d4cf827eeb6b9c852c88c54a5a77d6ca34706d84f86"


def training_digest(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def train_session(root, local_root, enabled, per_class, extra_blocks):
    if not enabled:
        print("续训关闭。仅查看已有结果可直接运行第③格。")
        return
    if type(per_class) is not int or per_class not in (24, 48, 96, 192):
        raise ValueError("PER_CLASS 只能为 24、48、96 或 192")
    if type(extra_blocks) is not int or not 1 <= extra_blocks <= 10:
        raise ValueError("BLOCKS_THIS_SESSION 必须是 1～10 的整数；它不改变每档总预算。")
    work = local_root / "hanzi81-e477bdd4894e"
    data = work / "training/data/baseline81-v2"
    suite = root / "analyses/baseline81_s42_v1/03_learning_curve/steps2430_s42_v1"
    plan_path = suite / "plan.json"
    for path in [plan_path, data / "train.npz", data / "val_select.npz"]:
        if not path.is_file():
            raise FileNotFoundError(f"缺少 {path}。请先运行第①格恢复 Drive、代码和数据。")
    plan = json.loads(plan_path.read_text(encoding="utf-8"))
    if (plan["curve_script_sha256"], plan["blocks"], plan["block_draws"],
            plan["batch_size"], plan["seed"], plan["sizes"]) != (
            TRAIN_CURVE_SHA256, 10, 15552, 64, 42, [24, 48, 96, 192]):
        raise ValueError("冻结实验协议不同；未启动训练。")
    # 本格拥有全部导入与路径，不调用其他单元临时定义的辅助函数。
    source = suite / "learning_curve.py"
    expected_files = {source: TRAIN_CURVE_SHA256,
                      work / "training/train.py": TRAIN_SOURCE_SHA256,
                      work / "training/io_utils.py": TRAIN_IO_SHA256,
                      data / "manifest.json": TRAIN_DATA_SHA256}
    for path, expected in expected_files.items():
        if not path.is_file() or training_digest(path) != expected:
            raise ValueError(f"缺失或校验失败：{path}。请运行第①格检查。")
    script = local_root / "hanzi81_learning_curve.py"
    if not script.is_file() or training_digest(script) != TRAIN_CURVE_SHA256:
        script.write_bytes(source.read_bytes())
    if training_digest(script) != TRAIN_CURVE_SHA256:
        raise ValueError("恢复学习曲线脚本失败")
    command = [sys.executable, "-u", str(script), "train", "--work", str(work),
               "--data", str(data), "--root", str(root), "--sizes", str(per_class),
               "--stop-after-blocks", str(extra_blocks)]
    log_dir = suite / "session_logs"
    log_dir.mkdir(exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
    log_path = log_dir / f"n{per_class:03d}_{stamp}.txt"
    print(f"本次：每字 {per_class} 图，最多追加 {extra_blocks} 块；总预算仍是 10 块。", flush=True)
    print("已完成档位由原脚本校验后跳过。日志：", log_path, flush=True)
    with log_path.open("w", encoding="utf-8") as log:
        log.write(json.dumps({"command": command, "plan_sha256": training_digest(plan_path)},
                             ensure_ascii=False) + "\n")
        log.flush()
        with subprocess.Popen(command, cwd=work, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding="utf-8", errors="replace", bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                returncode = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                print("进程已停止；下次从最近成功保存的完整块恢复。请运行第③格检查。")
                raise
    if returncode:
        raise RuntimeError(f"训练退出码 {returncode}；请保留并反馈日志：{log_path}")
    arm = suite / f"n{per_class:03d}_s42"
    required = [arm / name for name in ("config.json", "history.json", "best.pt", "last.pt")]
    if any(not path.is_file() or path.stat().st_size == 0 for path in required):
        raise RuntimeError("本次进程已退出，但检查点文件不完整；请先检查保存情况。")
    history = json.loads((arm / "history.json").read_text(encoding="utf-8"))
    if not history or [r["block"] for r in history] != list(range(1, len(history) + 1)):
        raise ValueError("训练日志不是连续完整块，请保留现场检查。")
    print(f"本档已记录 {len(history)}/10 个完整块；本次分段正常结束。")
    print("运行第③格查看四档进度。未完成全部档位属于正常状态。")
    print("确认 Drive 文件保存后，可断开连接并删除运行时；不要同时在两个会话续训同一档位。")


if __name__ == "__main__":
    train_session(DRIVE_ROOT, LOCAL_ROOT, RUN_TRAINING, PER_CLASS, BLOCKS_THIS_SESSION)


## ③ 检查保存结果与已有报告

可在全新 CPU 会话中单独运行：只挂载 Drive、读 JSON、检查文件存在性并绘图；
不导入 torch、不读取训练图像、不执行模型推理。没有第①/②格的变量也可以运行。

第 1/2 项和开发集指标从已保存文件读取。第 3 项逐档显示实际日志进度，
尚未完成时只画进度图；四档都有完整预算记录及检查点文件时才画最终比较图。
第 6 项读取已有三轮计时并检验内部一致性；成功时完成首轮效率分析。
测量值与估算分别列示：实际 compute units、费用、GPU 利用率、CPU/RAM 峰值没有测量，
保持为空；显存 allocated/reserved 不相加，也不当作 GPU 计算利用率。
1140 字时长只是保持每类样本配比的线性排期估算，不代表已扩类训练或确定训练轮数。

默认在 `analyses/baseline81_s42_v1/session_reports/<时间戳>/` 新建本次报告，
不覆盖原有分析。这里不加载模型权重，检查点内部状态仍由原续训脚本验证；
挂载读取成功不等于独立验证云端上传已完成。

确认日志和 Drive 文件保存后，选择 **断开连接并删除运行时**。
下次仅读报告不需要 GPU；额度耗尽也不必重跑已经完成的实验。


In [ ]:
#@title ③ 结果检查与汇总（可单独运行，无需 GPU）
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import statistics
import uuid
"""Analyse saved three-epoch T4 measurements using only the Python standard library.

The notebook generator embeds this file into cell 3; it does not run profiling.
"""
import math
import statistics


def efficiency_positive(value, name, allow_zero=False):
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
        raise ValueError(f"计时报告字段无效：{name}")
    if value < 0 or (not allow_zero and value == 0):
        raise ValueError(f"计时报告字段应为正数：{name}")
    return float(value)


def efficiency_equal(actual, expected, name):
    if not math.isclose(actual, expected, rel_tol=1e-8, abs_tol=1e-8):
        raise ValueError(f"计时报告内部不一致：{name}")


def analyze_efficiency(profile, baseline=None, reference_environment=None):
    """Validate measurements, derive ratios, and label assumptions explicitly."""
    if profile["purpose"] != "runtime_profile_only_not_scientific_results":
        raise ValueError("不是原独立计时实验报告")
    if profile["batch_size"] != 64 or profile["environment"]["gpu"] != "Tesla T4":
        raise ValueError("当前分析对应原 T4 / batch=64 计时实验")
    if reference_environment is not None and profile["environment"] != reference_environment:
        raise ValueError("计时报告环境与基线配置不一致")
    epochs = profile["epochs"]
    if [r["epoch"] for r in epochs] != [1, 2, 3]:
        raise ValueError("需要完整的三轮计时报告；不将部分测量标记完成")
    loads = {r["split"]: r for r in profile["dataset_loads"]}
    if loads["train"]["samples"] != 15552 or loads["val_select"]["samples"] != 7776:
        raise ValueError("计时报告数据规模与 81 字基线不一致")
    for value in loads.values():
        efficiency_positive(value["seconds"], "dataset_load_seconds", allow_zero=True)
    timing_keys = ("train_seconds", "validation_seconds", "checkpoint_save_seconds",
                   "epoch_wall_seconds_including_saves")
    rate_keys = ("train_images_per_second", "validation_images_per_second")
    total_memory = efficiency_positive(profile["memory_before"]["device_total_gib"], "device_total_gib")
    for row in epochs:
        for key in timing_keys + rate_keys + ("peak_allocated_gib", "peak_reserved_gib"):
            efficiency_positive(row[key], key)
        efficiency_equal(row["train_images_per_second"], 15552 / row["train_seconds"], "train throughput")
        efficiency_equal(row["validation_images_per_second"], 7776 / row["validation_seconds"], "validation throughput")
        if not row["peak_allocated_gib"] <= row["peak_reserved_gib"] <= total_memory:
            raise ValueError("显存统计关系异常")
        stages = row["train_seconds"] + row["validation_seconds"] + row["checkpoint_save_seconds"]
        if stages > row["epoch_wall_seconds_including_saves"] + .1:
            raise ValueError("阶段耗时超过整轮 wall time")
        saves = [s for s in profile["checkpoint_saves"] if s["epoch"] == row["epoch"]]
        if sorted(s["file"] for s in saves) != sorted(row["saved_files"]) or "last.pt" not in row["saved_files"]:
            raise ValueError("检查点保存记录不完整")
        for save in saves:
            efficiency_positive(save["size_mib"], "checkpoint size")
            efficiency_positive(save["seconds"], "checkpoint time", allow_zero=True)
        efficiency_equal(sum(s["seconds"] for s in saves), row["checkpoint_save_seconds"], "checkpoint total")
    steady = {key: statistics.median(row[key] for row in epochs[1:]) for key in timing_keys + rate_keys}
    for key, value in steady.items():
        efficiency_equal(profile["median_of_epochs_2_and_3"][key], value, key)
    allocated = max(row["peak_allocated_gib"] for row in epochs)
    reserved = max(row["peak_reserved_gib"] for row in epochs)
    efficiency_equal(profile["maximum_allocated_gib"], allocated, "maximum allocated")
    efficiency_equal(profile["maximum_reserved_gib"], reserved, "maximum reserved")
    call_seconds = efficiency_positive(profile["three_epoch_train_call_seconds_including_setup"], "three_epoch_call")
    if call_seconds + .2 < sum(r["epoch_wall_seconds_including_saves"] for r in epochs):
        raise ValueError("三轮总调用时长小于逐轮 wall time 之和")
    wall = steady["epoch_wall_seconds_including_saves"]
    phases = {"train": steady["train_seconds"], "validation": steady["validation_seconds"],
              "checkpoint_save": steady["checkpoint_save_seconds"]}
    phases["other_and_rounding"] = wall - sum(phases.values())
    phase_percent = {k: value / wall * 100 for k, value in phases.items()}
    checkpoint_sizes = {name: max(s["size_mib"] for s in profile["checkpoint_saves"] if s["file"] == name)
                        for name in ("best.pt", "last.pt")}
    ratio = 1140 / 81
    measured_compute = steady["train_seconds"] + steady["validation_seconds"]
    observed_baseline = None
    if baseline and "completed_epochs" in baseline:
        observed_baseline = {k: baseline[k] for k in (
            "completed_epochs", "planned_epochs", "compute_minutes_excluding_saves",
            "steady_median_seconds", "thresholds") if k in baseline}
    return {
        "schema": 1, "status": "complete_saved_measurement_analysis",
        "scope": "81-class T4 batch64 runtime analysis; no new training or model evaluation",
        "environment": profile["environment"], "batch_size": 64,
        "measured_epochs": 3, "steady_epochs": [2, 3],
        "samples_per_epoch": {"train": 15552, "validation": 7776},
        "steady": steady,
        "steady_observed_epoch_wall_range_seconds": {
            "min": min(r["epoch_wall_seconds_including_saves"] for r in epochs[1:]),
            "max": max(r["epoch_wall_seconds_including_saves"] for r in epochs[1:])},
        "phase_seconds": phases,
        "phase_percent_of_steady_wall": phase_percent,
        "memory": {"device_total_gib": total_memory, "peak_allocated_gib": allocated,
                   "peak_reserved_gib": reserved, "allocated_percent_of_capacity": allocated / total_memory * 100,
                   "reserved_percent_of_capacity": reserved / total_memory * 100,
                   "reserved_includes_allocated": True, "gpu_utilization_percent": None},
        "storage": {"checkpoint_sizes_mib": checkpoint_sizes,
                    "best_plus_last_mib": sum(checkpoint_sizes.values()),
                    "four_pairs_estimate_mib": 4 * sum(checkpoint_sizes.values()),
                    "notes": "Four-pair estimate excludes data, logs, extra runs and temporary checkpoint files."},
        "measured_setup": {"dataset_load_seconds": sum(r["seconds"] for r in loads.values()),
                           "three_epoch_train_call_seconds_including_setup": call_seconds},
        "baseline_history_observation": observed_baseline,
        "planning_estimates_not_measurements": {
            "same_81_class_40_epochs_minutes_including_saves": wall * 40 / 60,
            "one_learning_curve_arm_10_blocks_minutes": wall * 10 / 60,
            "four_learning_curve_arms_40_blocks_minutes": wall * 40 / 60,
            "learning_curve_note": "Same 15552 train + 7776 validation draws per block; extra curve metrics/setup overhead unmeasured.",
            "hypothetical_1140_classes": {
                "train_images": 1140 * 192, "validation_images": 1140 * 96,
                "sample_count_ratio": ratio,
                "train_and_validation_seconds_per_epoch": measured_compute * ratio,
                "train_and_validation_hours_for_40_epochs": measured_compute * ratio * 40 / 3600,
                "assumptions": "Keep 192 train/96 validation per class, 64x64 input, batch64 and T4; assume time proportional to sample count. Larger classifier, data pipeline, saves, setup and convergence are unmeasured. This does not authorize expansion or set its epoch budget."}},
        "conclusions": [
            f"当前测量下训练占整轮 {phase_percent['train']:.2f}%，验证占 {phase_percent['validation']:.2f}%，检查点保存占 {phase_percent['checkpoint_save']:.2f}%。",
            ("保存占比较低，现有数据不支持为省时降低断点保存频率。" if phase_percent['checkpoint_save'] < 5
             else "保存开销值得单独排查；调整保存频率前仍需考虑中断损失。"),
            "显存容量占比不等于 GPU 计算利用率，也不直接证明增大 batch 会提速。",
            "保持当前实验 batch=64；后续吞吐优化若改变 batch、精度或数据流程，应另做独立基准。",
            "81 字开发准确率的收敛观察不能直接用来确定 1140 字的训练轮数。"],
        "not_measured": {"compute_units_consumed": None, "colab_cost": None, "gpu_utilization": None,
                         "cpu_utilization": None, "system_ram_peak": None,
                         "original_40_epoch_wall_time_including_saves": None},
        "limitations": [
            "独立临时模型的三轮计时；稳态仅用第 2、3 轮，不给统计置信区间。",
            "显存是计时子进程的 PyTorch allocated/reserved，reserved 已含 allocated，不可相加。",
            "阶段占比使用同一整轮 wall time 分母；余项包含其他开销和整轮 0.1 秒舍入。",
            "本地数据载入时间受缓存影响；不是 Drive 下载速度或完整初始化耗时。",
            "Drive 保存计时到挂载文件系统调用返回，不证明远端上传已完成。",
            "样本量外推只是排期估算；不能替代扩类后实测或推断泛化表现。"]}


def print_efficiency(analysis):
    steady, mem = analysis["steady"], analysis["memory"]
    print("\n第 6 项：训练效率与资源分析（复用已有测量，无新增 GPU 训练）")
    print(f"整轮含保存：{steady['epoch_wall_seconds_including_saves']:.3f} s")
    print(f"训练吞吐：{steady['train_images_per_second']:.2f} 图/s；验证吞吐：{steady['validation_images_per_second']:.2f} 图/s")
    names = {"train": "训练", "validation": "验证", "checkpoint_save": "检查点保存", "other_and_rounding": "其他及舍入"}
    for key, seconds in analysis["phase_seconds"].items():
        print(f"  {names[key]}：{seconds:.3f} s / {analysis['phase_percent_of_steady_wall'][key]:.2f}%")
    print(f"峰值张量占用：{mem['peak_allocated_gib']:.3f} GiB ({mem['allocated_percent_of_capacity']:.2f}% 容量)")
    print(f"峰值预留：{mem['peak_reserved_gib']:.3f} GiB ({mem['reserved_percent_of_capacity']:.2f}% 容量，已含张量占用)")
    print(f"best + last：{analysis['storage']['best_plus_last_mib']:.2f} MiB；不含临时文件等空间。")
    estimate = analysis["planning_estimates_not_measurements"]
    print(f"81 字 40 轮含保存排期估算：{estimate['same_81_class_40_epochs_minutes_including_saves']:.2f} 分钟（非原训练实测 wall time）。")
    full = estimate["hypothetical_1140_classes"]
    print(f"假设 1140 字保持每类 192/96 图，40 轮训练+验证线性估算：{full['train_and_validation_hours_for_40_epochs']:.2f} 小时；不含保存等开销，扩类后需实测。")
    print("实际 compute units、费用、GPU 利用率、CPU/RAM 峰值：未测量。")
    for text in analysis["conclusions"]:
        print("-", text)


def plot_efficiency(analysis, out=None, show=True):
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    phases = analysis["phase_seconds"]
    axes[0].barh(["Train", "Validation", "Checkpoint", "Other + rounding"], list(phases.values()),
                 color=["#3676b6", "#e38d36", "#4d9971", "#87929c"])
    axes[0].invert_yaxis()
    axes[0].set_xlabel("Seconds per epoch (median of epochs 2 and 3)")
    mem = analysis["memory"]
    axes[1].barh(["Peak allocated", "Peak reserved (includes allocated)", "Device capacity"],
                 [mem["peak_allocated_gib"], mem["peak_reserved_gib"], mem["device_total_gib"]],
                 color=["#3676b6", "#7398bd", "#c4cbd1"])
    axes[1].invert_yaxis()
    axes[1].set_xlabel("GiB (memory capacity, not GPU utilization)")
    for ax in axes:
        ax.grid(axis="x", alpha=.2)
        ax.set_axisbelow(True)
    fig.tight_layout()
    if out:
        fig.savefig(out / "efficiency_resources.png", dpi=160)
    if show:
        plt.show()
    plt.close(fig)


SAVE_REPORT = True #@param {type:"boolean"}
DRIVE_ROOT = Path("/content/drive/MyDrive/hanzi81")


def report_digest(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def report_read(path):
    if not path.is_file():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, ValueError) as exc:
        raise RuntimeError(f"报告无法读取；未自动覆盖或重算：{path}") from exc


def report_files(folder, names):
    return {name: {"exists": (folder / name).is_file(),
                   "bytes": (folder / name).stat().st_size if (folder / name).is_file() else None}
            for name in names}


def report_efficiency(profile_path, baseline, run):
    if profile_path is None:
        return {"status": "incomplete_missing_measurement", "message": "未找到三轮计时报告；未自动重跑。"}
    try:
        profile = report_read(profile_path)
        reference = report_read(run / "config.json")
        reference_environment = reference["environment"] if reference is not None else None
        analysis = analyze_efficiency(profile, baseline, reference_environment)
        protocol = report_read(profile_path.parent / "protocol.json")
        frozen_script = profile_path.parent / "profile_runtime.py"
        expected_script = "131d07fadea99eece5d0c1c4179e454b8149f8ec960a3c6ba2229f40886052bc"
        if frozen_script.is_file() and report_digest(frozen_script) != expected_script:
            raise ValueError("计时脚本哈希与原版本不同")
        if protocol is not None:
            if protocol["profiler_sha256"] != expected_script or protocol["measured_epochs"] != 3:
                raise ValueError("计时协议与原版本不同")
            if reference is not None:
                for key in ("data_sha256", "trainer_sha256", "environment"):
                    if protocol[key] != reference[key]:
                        raise ValueError(f"计时协议与原基线不匹配：{key}")
        analysis["source"] = {"summary_path": str(profile_path), "summary_sha256": report_digest(profile_path),
                              "baseline_config_checked": reference is not None,
                              "protocol_checked": protocol is not None,
                              "frozen_profiler_checked": frozen_script.is_file(),
                              "new_gpu_measurement": False}
        return analysis
    except (KeyError, TypeError, ValueError, RuntimeError) as exc:
        return {"status": "incomplete_invalid_measurement", "message": str(exc),
                "source_path": str(profile_path), "new_gpu_measurement": False}


def report_learning_progress(suite):
    plan = report_read(suite / "plan.json")
    if plan is not None and (plan["blocks"], plan["block_draws"], plan["batch_size"], plan["sizes"]) != (
            10, 15552, 64, [24, 48, 96, 192]):
        raise ValueError("学习曲线协议与当前 Notebook 不同；不混合比较。")
    rows, histories = [], {}
    for size in (24, 48, 96, 192):
        folder = suite / f"n{size:03d}_s42"
        history = report_read(folder / "history.json")
        files = report_files(folder, ("config.json", "history.json", "best.pt", "last.pt"))
        artifacts_exist = all(value["exists"] and value["bytes"] > 0 for value in files.values())
        row = {"per_class": size, "unique_training_images": size * 81,
               "logged_blocks": None, "planned_blocks": 10,
               "complete_by_history_and_file_presence": False, "files": files,
               "status": "未找到训练日志；不据此断言从未训练"}
        if history is not None:
            if [r["block"] for r in history] != list(range(1, len(history) + 1)) or len(history) > 10:
                raise ValueError(f"{folder.name} 日志块编号异常")
            for record in history:
                if record["batch_attempts"] != record["block"] * 243 or record["validation_split"] != "val_select":
                    raise ValueError(f"{folder.name} 日志与固定批次预算不同")
            row.update(logged_blocks=len(history),
                       complete_by_history_and_file_presence=len(history) == 10 and artifacts_exist,
                       history_sha256=report_digest(folder / "history.json"))
            if history:
                cfg = report_read(folder / "config.json")
                if plan is None or cfg is None or cfg["suite_plan_sha256"] != report_digest(suite / "plan.json"):
                    raise ValueError(f"{folder.name} 缺少匹配的冻结协议或配置")
                if cfg["unique_per_class"] != size or cfg["blocks"] != 10 or cfg["seed"] != 42:
                    raise ValueError(f"{folder.name} 配置不匹配")
                best = max(history, key=lambda r: (r["validation"]["macro_accuracy"],
                                                    -r["validation"]["loss"], -r["block"]))
                row.update(last_completed_validation=history[-1]["validation"],
                           best_validation=best["validation"], best_block=best["block"],
                           optimizer_updates=history[-1]["optimizer_updates"],
                           amp_skipped_updates=sum(r["amp_skipped_updates_this_block"] for r in history))
                row["status"] = ("已记录完整预算" if len(history) == 10 else "中间进度，待续训") if artifacts_exist else "文件不完整，需检查"
            else:
                row["status"] = "空日志，尚无已记录完整块"
            histories[size] = history
        rows.append(row)
    return {"plan_found": plan is not None, "results": rows,
            "all_complete_by_history_and_file_presence": all(r["complete_by_history_and_file_presence"] for r in rows),
            "checkpoint_contents_loaded": False,
            "note": "这里只检查日志与模型文件存在性；未反序列化权重。实际续训由原脚本验证检查点。"}, histories


def report_baseline_history(run):
    history = report_read(run / "history.json")
    if history is None:
        return {"status": "未找到基线 history.json"}, []
    if not history or [r["epoch"] for r in history] != list(range(1, len(history) + 1)):
        raise ValueError("基线日志轮次不连续或为空")
    if len(history) > 40 or any(r["validation_split"] != "val_select" for r in history):
        raise ValueError("基线日志协议不匹配")
    selected = max(history, key=lambda r: (r["validation"]["macro_accuracy"],
                                           -r["validation"]["loss"], -r["epoch"]))
    thresholds = {}
    for name, threshold in [("99%", .99), ("99.9%", .999), ("100%", 1.0)]:
        passed = [r["validation"]["macro_accuracy"] >= threshold for r in history]
        thresholds[name] = {
            "first_epoch": next((history[i]["epoch"] for i in range(len(history)) if passed[i]), None),
            "sustained_through_last_logged_epoch_from": next(
                (history[i]["epoch"] for i in range(len(history)) if all(passed[i:])), None)}
    times = [r["seconds_compute"] for r in history]
    return {"completed_epochs": len(history), "planned_epochs": 40,
            "history_sha256": report_digest(run / "history.json"),
            "selected_epoch_by_original_rule": selected["epoch"],
            "selected_validation": selected["validation"], "last_validation": history[-1]["validation"],
            "thresholds": thresholds, "compute_minutes_excluding_saves": sum(times) / 60,
            "steady_median_seconds": statistics.median(times[1:]) if len(times) > 1 else None,
            "files": report_files(run, ("config.json", "history.json", "best.pt", "last.pt"))}, history


def report_plots(baseline, progress, histories, out):
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("当前环境没有 matplotlib；JSON 汇总仍可使用，不自动安装依赖。")
        return
    if baseline:
        fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
        epochs = [r["epoch"] for r in baseline]
        for key, label in [("train", "train"), ("validation", "val_select")]:
            axes[0].plot(epochs, [max(r[key]["loss"], 1e-12) for r in baseline], label=label)
            axes[1].plot(epochs, [100 * r[key]["accuracy"] for r in baseline], label=label)
        axes[0].set_yscale("log")
        axes[0].set_ylabel("Cross entropy")
        axes[1].set_ylabel("Accuracy (%)")
        axes[2].plot(epochs, [r["seconds_compute"] for r in baseline])
        axes[2].set_ylabel("Train + validation (s)")
        for ax in axes:
            ax.set_xlabel("Epoch")
            ax.grid(alpha=.3)
        axes[0].legend()
        axes[1].legend()
        fig.tight_layout()
        if out:
            fig.savefig(out / "baseline_history.png", dpi=160)
        plt.show()
        plt.close(fig)
    if any(histories.values()):
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
        for size, history in histories.items():
            if history:
                batches = [r["batch_attempts"] for r in history]
                axes[0].plot(batches, [100 * r["validation"]["macro_accuracy"] for r in history], label=f"n={size}")
                axes[1].plot(batches, [max(r["validation"]["loss"], 1e-12) for r in history], label=f"n={size}")
        axes[0].set_ylabel("val_select macro accuracy (%)")
        axes[1].set_ylabel("val_select cross entropy")
        axes[1].set_yscale("log")
        for ax in axes:
            ax.set_xlabel("Completed batch attempts")
            ax.grid(alpha=.3)
            ax.legend()
        fig.suptitle("Learning-curve progress (partial budgets may differ)")
        fig.tight_layout()
        if out:
            fig.savefig(out / "learning_curve_progress.png", dpi=160)
        plt.show()
        plt.close(fig)
    if progress["all_complete_by_history_and_file_presence"]:
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
        rows = progress["results"]
        for key, label in [("last_completed_validation", "Final: fixed budget"),
                           ("best_validation", "Selected best: secondary")]:
            x = [r["per_class"] for r in rows]
            axes[0].plot(x, [100 * r[key]["macro_accuracy"] for r in rows], "o-", label=label)
            axes[1].plot(x, [max(r[key]["loss"], 1e-12) for r in rows], "o-", label=label)
        axes[0].set_ylabel("val_select macro accuracy (%)")
        axes[1].set_ylabel("val_select cross entropy")
        axes[1].set_yscale("log")
        for ax in axes:
            ax.set_xlabel("Unique training images per class")
            ax.set_xticks([24, 48, 96, 192])
            ax.grid(alpha=.3)
            ax.legend()
        fig.tight_layout()
        if out:
            fig.savefig(out / "learning_curve_fixed_budget.png", dpi=160)
        plt.show()
        plt.close(fig)
    else:
        print("四档尚无完整预算结果：仅展示进度，不生成最终固定预算比较图。")


def inspect_session(root, save_report):
    # 本格可以在新 CPU 会话直接执行，不依赖第①/②格、torch 或 /content 中的数据。
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive
        drive.mount("/content/drive")
    if not root.is_dir():
        raise FileNotFoundError(f"请检查 Drive 项目路径：{root}")
    run = root / "runs/baseline81_s42_v1"
    analyses = root / "analyses/baseline81_s42_v1"
    baseline, history = report_baseline_history(run)
    progress, histories = report_learning_progress(analyses / "03_learning_curve/steps2430_s42_v1")
    existing = {}
    paths = {"01_history": analyses / "01_history/summary.json",
             "02_margins": analyses / "02_margins/summary.json",
             "val_select": run / "evaluation_val_select/metrics.json",
             "val_graph": run / "evaluation_val_graph/metrics.json"}
    profiles = sorted((analyses / "06_profile").glob("*/summary.json"))
    if profiles:
        paths["06_profile_saved_measurement"] = profiles[-1]
    for key, path in paths.items():
        value = report_read(path)
        existing[key] = {"path": str(path), "found": value is not None,
                         "sha256": report_digest(path) if value is not None else None, "report": value}
    efficiency = report_efficiency(profiles[-1] if profiles else None, baseline, run)
    snapshot = {
        "purpose": "read_saved_files_only_no_model_evaluation",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "baseline": baseline, "learning_curve": progress, "existing_reports": existing,
        "task_06_status": efficiency["status"], "efficiency_resources": efficiency,
        "notes": ["第 6 项根据已保存测量的实际校验与分析结果更新；不新增 GPU 训练。",
                  "本单元未启动训练、模型推理或 test 评估，也未反序列化检查点。",
                  "Drive 挂载读取成功不等于已独立验证远端上传持久性。"]}
    print("基线日志：", json.dumps(baseline, ensure_ascii=False, indent=2))
    print("\n学习曲线：每字图数 | 日志完整块/10 | 状态")
    for row in progress["results"]:
        count = row["logged_blocks"] if row["logged_blocks"] is not None else "未知"
        print(f"{row['per_class']:>3} | {count}/10 | {row['status']}")
        if "last_completed_validation" in row:
            print("  最近验证：", row["last_completed_validation"])
    print("\n已有报告：")
    for key, value in existing.items():
        print(key, "已找到" if value["found"] else "未找到", value["path"])
    if efficiency["status"] == "complete_saved_measurement_analysis":
        print_efficiency(efficiency)
    else:
        print("第 6 项仍未完成：", efficiency["message"])
    out = None
    if save_report:
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
        out = analyses / "session_reports" / stamp
        out.mkdir(parents=True, exist_ok=False)
        (out / "session_summary.json").write_text(
            json.dumps(snapshot, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        print("本次检查报告：", out / "session_summary.json")
        (out / "efficiency_resources.json").write_text(
            json.dumps(efficiency, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        print("第 6 项分析文件：", out / "efficiency_resources.json")
    report_plots(history, progress, histories, out)
    if efficiency["status"] == "complete_saved_measurement_analysis":
        plot_efficiency(efficiency, out)
    print("模型权重未在本格加载；续训时仍需原脚本验证。确认文件保存后，可释放 GPU 运行时。")
    return snapshot


if __name__ == "__main__":
    inspect_session(DRIVE_ROOT, SAVE_REPORT)
